In [ ]:
# If imbalanced-learn is not installed, uncomment the next line and run it once.
# !pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("customers-10000.csv")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(list(df.columns))

# Partition

In [ ]:
split_fraction = 0.70
partition = df.sample(frac=split_fraction, random_state=42)
print("Partition size:", len(partition))
partition.head()

# Random

In [ ]:
sample_random = df.sample(n=100, random_state=42)
sample_random.head()

# Stratified

In [ ]:
sample_stratified = (
    df.groupby("Country", group_keys=False)
      .apply(lambda x: x.sample(frac=0.10, random_state=42))
      .reset_index(drop=True)
)
print("Stratified sample size:", len(sample_stratified))
sample_stratified.head()

## Task 1: Fix Number of Sample (500 only)

In [ ]:
# Take exactly 500 observations from the complete dataset
sample_500 = df.sample(n=500, random_state=42)
print("Sample size:", len(sample_500))
sample_500.head()

## Task 2: Country-wise list/count

In [ ]:
country_counts = df["Country"].value_counts().reset_index()
country_counts.columns = ["Country", "Count"]
country_counts

## Task 3: Equalize the country wise count

In [ ]:
# Use the smallest country count as the common sample size
min_country_count = df["Country"].value_counts().min()

sample_equal = (
    df.groupby("Country", group_keys=False)
      .apply(lambda x: x.sample(n=min_country_count, random_state=42))
      .reset_index(drop=True)
)

print("Records per country after equalization:")
print(sample_equal["Country"].value_counts())
sample_equal.head()

# Cluster

In [ ]:
countries = df["Country"].dropna().unique()
selected_countries = pd.Series(countries).sample(n=min(2, len(countries)), random_state=42)
sample_cluster = df[df["Country"].isin(selected_countries)]
print("Selected countries:", selected_countries.tolist())
sample_cluster.head(36)

## Task 4: Sample from 5 most/least listed countries

In [ ]:
country_counts = df["Country"].value_counts()
most_listed = country_counts.head(5).index.tolist()
least_listed = country_counts.tail(5).index.tolist()
selected = most_listed + least_listed

# Take up to 10 records from each selected country
sample_most_least = (
    df[df["Country"].isin(selected)]
      .groupby("Country", group_keys=False)
      .apply(lambda x: x.sample(n=min(10, len(x)), random_state=42))
      .reset_index(drop=True)
)

print("5 most listed countries:", most_listed)
print("5 least listed countries:", least_listed)
print("\nSample count by country:")
print(sample_most_least["Country"].value_counts())
sample_most_least.head()

# Data Selection

1. Missing Value
2. Identical Values
3. Correlation Analysis
4. Chi-Square Test

In [ ]:
# 1. Missing value analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Missing values:")
print(missing if len(missing) else "No missing values found")

# 2. Identical/duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# 3. Correlation analysis for numeric variables
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()
    plt.figure(figsize=(10, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Matrix")
    plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")

In [ ]:
# 4. Chi-Square test for categorical variables against Fraud
from scipy.stats import chi2_contingency

if "Fraud" in df.columns:
    categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    categorical_cols = [c for c in categorical_cols if c != "Fraud"]
    chi_results = []

    for col in categorical_cols:
        table = pd.crosstab(df[col], df["Fraud"])
        if table.shape[0] > 1 and table.shape[1] > 1:
            chi_stat, p_value, _, _ = chi2_contingency(table)
            chi_results.append([col, chi_stat, p_value])

    chi_results = pd.DataFrame(chi_results, columns=["Feature", "Chi-Square", "p-value"])
    chi_results = chi_results.sort_values("p-value")
    chi_results.head(10)
else:
    print("Fraud column is not available in the dataset.")

In [ ]:
# Prepare X and Y for the ML model
X = df.drop(columns=["Fraud"])
Y = df["Fraud"].copy()

if Y.dtype == "object" or str(Y.dtype) == "category":
    target_encoder = LabelEncoder()
    Y = target_encoder.fit_transform(Y.astype(str))
else:
    Y = Y.astype(int)

print("Target distribution:")
print(pd.Series(Y).value_counts())

# Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Task 5: ML Model

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Balance only the training data
ros = RandomOverSampler(random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_processed, y_train)

print("Before oversampling:", pd.Series(y_train).value_counts().to_dict())
print("After oversampling:", pd.Series(y_train_balanced).value_counts().to_dict())

model = LogisticRegression(max_iter=1000)
model.fit(X_train_balanced, y_train_balanced)

y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1] if len(np.unique(y_train_balanced)) == 2 else None

## Result Analysis

In [ ]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

if y_prob is not None and len(np.unique(y_test)) == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

### Observation
The model should be judged using more than accuracy because fraud data can be imbalanced. Precision, recall, F1-score and ROC-AUC give a better view of how well fraudulent cases are being identified. Random oversampling is applied only to the training data, keeping the test set untouched for a fair evaluation.